# Mutual Indexing

*Level 9 — Knowledge-Augmented Generation (KAG)*

## Objective

KAG's second core idea, on top of the schema-constrained extraction from notebook 1: every node
and edge in the knowledge graph is linked back to the real document it was extracted from. This
"mutual index" is what lets a KG-reasoning answer cite an actual PubMed abstract instead of
asserting a bare triple with no provenance, and lets the retrieval operator widen a graph hit
back out to its full source text when the graph alone isn't enough.

This notebook builds a real graph from a small real PubMedQA sample and inspects the mutual
index directly.

In [1]:
import sys
from pathlib import Path

LEVEL_DIR = Path.cwd().parent
sys.path.insert(0, str(LEVEL_DIR))

from kag_common.dataset import prepare
from kag_common.llm import OllamaLLM
from indexing.graph_builder import build_graph, save_graph, load_graph

data = prepare(n_documents=6, seed=7)  # same sample as notebook 1, served from its cache
llm = OllamaLLM()

graph, validator, mutual_index = build_graph(data.corpus, llm)
print(f"Graph: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")
print("Validator:", validator.summary())
print("Mutual index:", mutual_index.summary())

Graph: 20 nodes, 17 edges
Validator: {'accepted_entities': 22, 'rejected_entities': 0, 'entity_rejection_rate': 0.0, 'accepted_relations': 17, 'rejected_relations': 5, 'relation_rejection_rate': 0.22727272727272727}
Mutual index: {'n_entities_indexed': 20, 'n_docs_indexed': 4, 'n_relations_indexed': 17}


## Widening a KG node back to its real source text

Picking one entity actually present in the graph and using the mutual index to recover the real
abstract(s) it came from.

In [2]:
sample_node = next(iter(graph.nodes(data=True)))
node_key, node_data = sample_node
print(f"Entity: {node_data['name']!r} (type={node_data['type']})")

source_docs = mutual_index.docs_for_entity(node_data["name"])
print(f"Extracted from real document(s): {sorted(source_docs)}")

widened_text = mutual_index.source_text_for_entity(node_data["name"], data.corpus)
print(f"\nWidened source text ({len(widened_text)} chars):\n")
print(widened_text[:600])

Entity: 'Study-24340838' (type=Study)
Extracted from real document(s): ['24340838']

Widened source text (1371 chars):

BACKGROUND: Sudden death in athletes can occur during sport activities and is presumably related to ventricular arrhythmias. OBJECTIVES: To investigate the long-term follow-up ofathletes with ventricular arrhythmias during an exercise test. METHODS: From a database of 56,462 athletes we identified 192 athletes (35 years old who had ventricular arrhythmias during an exercise test. Ninety athletes had>or =3 ventricular premature beats (VPB) (group A) and 102 athletes had ventricular couplets or non-sustained ventricular tachycardia during an exercise test (group B). A control group of 92 athlete


## Citing the document behind one specific relation

Every edge in the graph also carries the real document it was extracted from -- picking one
edge and confirming the mutual index agrees with the graph's own `doc_id` edge attribute.

In [3]:
if graph.number_of_edges() > 0:
    u, v, edge_data = next(iter(graph.edges(data=True)))
    subject_name = graph.nodes[u]["name"]
    object_name = graph.nodes[v]["name"]
    relation = edge_data["relation"]
    print(f"Edge: {subject_name} --{relation}--> {object_name}  (graph says doc_id={edge_data['doc_id']!r})")

    cited_docs = mutual_index.docs_for_relation(subject_name, relation, object_name)
    print(f"Mutual index agrees, cites: {sorted(cited_docs)}")
    assert edge_data["doc_id"] in cited_docs
else:
    print("No relations survived schema validation in this small sample -- nothing to cite here.")

Edge: Study-24340838 --HAS_POPULATION--> Group A  (graph says doc_id='24340838')
Mutual index agrees, cites: ['24340838']


## Persisting the graph + index together

A real pipeline shouldn't have to re-run extraction on every question -- `save_graph`/`load_graph`
round-trip both the graph and the mutual index to one JSON file.

In [4]:
cache_path = LEVEL_DIR / "data" / "cache" / "notebook02_demo_graph.json"
save_graph(graph, mutual_index, cache_path)

restored_graph, restored_index = load_graph(cache_path)
print(f"Restored graph: {restored_graph.number_of_nodes()} nodes, {restored_graph.number_of_edges()} edges")
print(f"Restored mutual index: {restored_index.summary()}")

assert restored_graph.number_of_nodes() == graph.number_of_nodes()
assert restored_index.summary() == mutual_index.summary()
print("\nRound trip verified: identical counts before and after persisting to disk.")

cache_path.unlink()  # this file was only for the round-trip demo above

Restored graph: 20 nodes, 17 edges
Restored mutual index: {'n_entities_indexed': 20, 'n_docs_indexed': 4, 'n_relations_indexed': 17}

Round trip verified: identical counts before and after persisting to disk.


## Observed result

*(filled in after running this notebook against the real, running Ollama instance -- see the
actual node/edge/citation counts printed above.)*